# Frame against window decisions

SVM and RF decide per frame, the sequence models per 50-frame window. Here the frame decisions of SVM and RF are combined over the same windows, by majority vote and by mean score, and all four models are compared per window on the 80 splits.

In [ ]:
import os
import sys
import json
import time
import importlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import glob
import shutil

def find_code_dir():
    here = os.path.abspath("..")
    if os.path.exists(os.path.join(here, "common.py")):
        return here
    hits = glob.glob("/kaggle/input/**/robustness/common.py", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("robustness/common.py not found (locally or under /kaggle/input)")

CODE_DIR = find_code_dir()
sys.path.insert(0, CODE_DIR)
import common as cm
importlib.reload(cm)

RESULTS_DIR = os.path.abspath("results")
SCORES_DIR = os.path.join(RESULTS_DIR, "frame_scores")
FRAME_MODELS = ["svm", "rf"]
PROTOCOLS = ["lopo", "l2po"]
SAVE_SCORES = True
RESUME_FROM = None

def resume(results_dir, source):
    if not source:
        return
    for f in glob.glob(os.path.join(source, "*.jsonl")):
        dst = os.path.join(results_dir, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copyfile(f, dst)
    if os.path.isdir(os.path.join(source, "frame_scores")):
        shutil.copytree(os.path.join(source, "frame_scores"), os.path.join(results_dir, "frame_scores"),
                        dirs_exist_ok=True)
    print("resumed from", source)

os.makedirs(RESULTS_DIR, exist_ok=True)
resume(RESULTS_DIR, RESUME_FROM)
print("Python", sys.version.split()[0], "|", sys.executable)
print("data:", cm.ORIG_DATA_DIR)

## Step 1: Load the features

In [ ]:
X, y = cm.load_frame_features(cm.ORIG_DATA_DIR)
for c in cm.ALL_CLUSTERS:
    print(f"{c:14s} label={y[c]}  frames={len(X[c]):5d}  windows={len(cm.window_index(len(X[c]))):4d}")

## Step 2: SVM and RF per frame and per window

In [ ]:
for model in FRAME_MODELS:
    for protocol in PROTOCOLS:
        path = os.path.join(RESULTS_DIR, f"frame_{model}_{protocol}.jsonl")
        done = cm.load_done(path)
        splits = cm.make_splits(protocol)
        print(f"=== {model} / {cm.PROTOCOLS[protocol]}: {len(splits)} splits ({len(done)} already done)")
        t_start = time.time()
        for i, sp in enumerate(splits, 1):
            if sp["name"] in done:
                continue
            r = cm.evaluate_frame_split(model, sp, X, y, {"test": X},
                                        scores_dir=os.path.join(SCORES_DIR, protocol) if SAVE_SCORES else None)
            cm.append_result(path, r)
            t = r["test"]
            print(f"  [{i:2d}/{len(splits)}] {sp['name']:32s} frame={t['frame']['accuracy']:.4f}  "
                  f"vote={t['window_vote']['accuracy']:.4f}  mean-score={t['window_mean_score']['accuracy']:.4f}  "
                  f"({time.time() - t_start:.0f}s)")

## Step 3: Sanity check

The per-frame accuracy must reproduce the stored SVM and RF results.

In [ ]:
STORED = {"svm": "SVM", "rf": "RF", "cnn_lstm": "CNN_LSTM", "cnn_transformer": "CNN_Transformer"}

def stored(model, protocol):
    return cm.load_done(os.path.join(cm.PV2, STORED[model], "results", f"splits_{protocol}.jsonl"))

for model in FRAME_MODELS:
    for protocol in PROTOCOLS:
        res = cm.load_done(os.path.join(RESULTS_DIR, f"frame_{model}_{protocol}.jsonl"))
        base = stored(model, protocol)
        d = [abs(res[n]["test"]["frame"]["accuracy"] - base[n]["accuracy"]) for n in res if n in base]
        if d:
            print(f"{model:4s} {cm.PROTOCOLS[protocol]}: {len(d)} splits, max |frame accuracy - stored| = {max(d):.5f}")

## Step 4: Comparison per window

In [ ]:
def accs(source, model, protocol):
    if source == "deep":
        r = stored(model, protocol)
        return {n: r[n]["accuracy"] for n in r}
    res = cm.load_done(os.path.join(RESULTS_DIR, f"frame_{model}_{protocol}.jsonl"))
    return {n: res[n]["test"][source]["accuracy"] for n in res}

ROWS = [("SVM, per frame", "frame", "svm"), ("SVM, window vote", "window_vote", "svm"),
        ("SVM, window mean score", "window_mean_score", "svm"),
        ("RF, per frame", "frame", "rf"), ("RF, window vote", "window_vote", "rf"),
        ("RF, window mean score", "window_mean_score", "rf"),
        ("CNN-LSTM (window)", "deep", "cnn_lstm"), ("CNN-Transformer (window)", "deep", "cnn_transformer")]

table = {}
print(f"{'model':26s} {'LOPO (%)':>16s} {'L2PO (%)':>16s} {'all 80 (%)':>16s} {'worst':>7s} {'n':>4s}")
for label, source, model in ROWS:
    per = {p: accs(source, model, p) for p in PROTOCOLS}
    allv = [v for p in PROTOCOLS for v in per[p].values()]
    table[label] = per
    cells = [f"{100*np.mean(list(per[p].values())):6.2f} ± {100*np.std(list(per[p].values())):5.2f}"
             if per[p] else "       --       " for p in PROTOCOLS]
    tot = f"{100*np.mean(allv):6.2f} ± {100*np.std(allv):5.2f}" if allv else "       --       "
    worst = f"{100*min(allv):6.2f}" if allv else "  --  "
    print(f"{label:26s} {cells[0]} {cells[1]} {tot} {worst} {len(allv):4d}")

## Step 5: Paired tests

Wilcoxon signed-rank tests with Holm correction.

In [ ]:
def paired(a_label, b_label):
    a = {(p, n): v for p in PROTOCOLS for n, v in table[a_label][p].items()}
    b = {(p, n): v for p in PROTOCOLS for n, v in table[b_label][p].items()}
    keys = [k for k in a if k in b]
    if len(keys) < 6:
        return None
    da, db = np.array([a[k] for k in keys]), np.array([b[k] for k in keys])
    d = da - db
    p = float(stats.wilcoxon(da, db).pvalue) if np.any(d != 0) else 1.0
    return {"a": a_label, "b": b_label, "n": len(keys), "delta": 100 * d.mean(),
            "wins": int((d > 0).sum()), "losses": int((d < 0).sum()), "ties": int((d == 0).sum()), "p": p}

tests = [paired(a, b) for a in ("CNN-Transformer (window)", "CNN-LSTM (window)")
         for b in ("SVM, window vote", "RF, window vote")]
tests += [paired("SVM, window vote", "SVM, per frame"), paired("RF, window vote", "RF, per frame")]
tests = [t for t in tests if t]
if tests:
    order = np.argsort([t["p"] for t in tests]); run = 0.0
    for i, k in enumerate(order):
        run = max(run, min(1.0, (len(tests) - i) * tests[k]["p"])); tests[k]["p_holm"] = run
    for t in tests:
        print(f"{t['a']:26s} vs {t['b']:22s}  Δ = {t['delta']:+6.2f} pts  W/L/T = {t['wins']}/{t['losses']}/{t['ties']}"
              f"  p_Holm = {t['p_holm']:.2g}  (n = {t['n']})")

with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump({"table": {k: {p: v for p, v in per.items()} for k, per in table.items()}, "tests": tests}, f, indent=1)
print("\nwritten:", os.path.join(RESULTS_DIR, "summary.json"))

## Step 6: Figure

In [ ]:
show = ["SVM, per frame", "SVM, window vote", "RF, per frame", "RF, window vote",
        "CNN-LSTM (window)", "CNN-Transformer (window)"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
rng = np.random.default_rng(1)
for ax, p in zip(axes, PROTOCOLS):
    for i, label in enumerate(show):
        v = 100 * np.array(list(table[label][p].values()))
        if len(v) == 0:
            continue
        col = "#2a78d6" if label.startswith("CNN") else "#8a8984"
        ax.scatter(i + rng.uniform(-0.2, 0.2, len(v)), v, s=10, color=col, alpha=0.6)
        ax.plot([i - 0.3, i + 0.3], [v.mean()] * 2, color="#1f1f1e", lw=1.5)
    ax.set_xticks(range(len(show)))
    ax.set_xticklabels([s.replace(" (window)", "").replace(", ", "\n") for s in show], fontsize=8)
    ax.set_title(cm.PROTOCOLS[p]); ax.grid(axis="y", color="#e6e5e0")
axes[0].set_ylabel("accuracy on held-out clusters (%)")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_DIR, f"frame_vs_window.{ext}"), dpi=200, bbox_inches="tight")
plt.show()